In [1]:
import re
import pandas as pd

csv_path = "../runs_sweep_full/mnist_mlp_reg/ep800_lr_0.001_wd_0.0003_hd_2048-512_do_0_ls_0.05/FASHIONMNIST/full/seed_0/final_train/crh_pah_metrics_all_layers_core.csv"
df = pd.read_csv(csv_path)

# detect layer names like fc1, fc2, fc3
layers = sorted({
    c.split("_")[0]
    for c in df.columns
    if re.match(r"^fc\d+_(Ha_WTW|Hb_WWT)_", c)
})

print("Layers found:", layers)
print("Epochs found:", sorted(df["epoch"].unique()))

Layers found: ['fc1', 'fc2', 'fc3']
Epochs found: [np.int64(200), np.int64(400), np.int64(600), np.int64(800), np.int64(1000000000)]


In [5]:
def print_pair(df, layer, pair_name, title):
    print("=" * 90)
    print(f"{layer} :: {title} :: LINEAR")
    print("=" * 90)

    linear_cols = [
        "epoch",
        f"{layer}_{pair_name}_eps_lin",
        f"{layer}_{pair_name}_c_lin",
        f"{layer}_{pair_name}_rho_fro",
        f"{layer}_{pair_name}_pearson_correlation",
    ]
    linear = df[linear_cols].copy()
    linear.columns = ["Epoch", "eps_lin", "c_lin", "rho_fro", "pearson_corr"]
    print(linear.to_string(index=False))
    print()

    print("=" * 90)
    print(f"{layer} :: {title} :: POWER-LAW")
    print("=" * 90)

    power_cols = [
        "epoch",
        f"{layer}_{pair_name}_powerlaw_alpha",
        f"{layer}_{pair_name}_powerlaw_r2",
        f"{layer}_{pair_name}_pow_eps",
        f"{layer}_{pair_name}_pow_c",
    ]
    power = df[power_cols].copy()
    power.columns = ["Epoch", "alpha", "R2", "eps_pow", "c_pow"]
    print(power.to_string(index=False))
    print()

In [3]:
for layer in layers:
    print_pair(df, layer, "Ha_WTW", "BACKWARD CRH/PAH")
    print_pair(df, layer, "Hb_WWT", "FORWARD CRH/PAH")

fc1 :: BACKWARD CRH/PAH :: LINEAR
     Epoch  eps_lin    c_lin  rho_fro
       200 0.788311 4.536556 0.615278
       400 0.885116 5.434815 0.465370
       600 0.984361 3.660537 0.176164
       800 0.994525 2.722842 0.104503
1000000000 0.994529 2.719059 0.104464

fc1 :: BACKWARD CRH/PAH :: POWER-LAW
     Epoch    alpha       R2  eps_pow    c_pow
       200 0.460659 0.911921 0.599937 0.066335
       400 0.427744 0.897167 0.625052 0.091450
       600 0.347930 0.875493 0.984252 4.143454
       800 0.168272 0.741372 0.989766 4.030562
1000000000 0.168232 0.741329 0.989760 4.027679

fc1 :: FORWARD CRH/PAH :: LINEAR
     Epoch  eps_lin      c_lin  rho_fro
       200 0.733121 123.817451 0.680098
       400 0.864382  66.393301 0.502836
       600 0.975632   8.202172 0.219413
       800 0.966001   4.108846 0.258539
1000000000 0.966000   4.101524 0.258541

fc1 :: FORWARD CRH/PAH :: POWER-LAW
     Epoch    alpha       R2  eps_pow    c_pow
       200 0.758035 0.909623 0.142240 0.284382
       400 0.

In [6]:
last = df.sort_values("epoch").iloc[-1]

rows = []
for layer in layers:
    rows.append({
        "layer": layer,
        "back_eps_lin": last[f"{layer}_Ha_WTW_eps_lin"],
        "back_pearson": last[f"{layer}_Ha_WTW_pearson_correlation"],
        "back_alpha": last[f"{layer}_Ha_WTW_powerlaw_alpha"],
        "back_R2": last[f"{layer}_Ha_WTW_powerlaw_r2"],
        "back_eps_pow": last[f"{layer}_Ha_WTW_pow_eps"],
        "fwd_eps_lin": last[f"{layer}_Hb_WWT_eps_lin"],
        "fwd_pearson": last[f"{layer}_Hb_WWT_pearson_correlation"],
        "fwd_alpha": last[f"{layer}_Hb_WWT_powerlaw_alpha"],
        "fwd_R2": last[f"{layer}_Hb_WWT_powerlaw_r2"],
        "fwd_eps_pow": last[f"{layer}_Hb_WWT_pow_eps"],
    })

pd.DataFrame(rows)

,layer,back_eps_lin,back_pearson,back_alpha,back_R2,back_eps_pow,fwd_eps_lin,fwd_pearson,fwd_alpha,fwd_R2,fwd_eps_pow
0,fc1,0.994529,0.100200,0.168232,0.741329,0.989760,0.966000,0.258542,0.786529,0.919421,0.965673
1,fc2,0.998992,0.044825,0.688171,0.871763,0.982911,0.948709,0.315737,1.164640,0.991591,0.912647
2,fc3,0.997929,0.064283,0.250215,0.882009,0.987546,0.496912,0.867801,0.917584,0.889868,0.191311
